# E1 + E2: reproduce the submitted results, then radiomics without shape features

**Revision plan (approved 17 Oct 2026).** This notebook runs two prespecified steps:

- **E1 — reproduction.** Recompute the submitted internal results from the saved files and check them against
  `audit_outputs/sex_correction_primary_20260718T174220Z`. Then re-run the Week-5 radiomics recipe exactly and
  check it against the saved Week-5 predictions. If E1 fails, the notebook stops before E2.
- **E2 — radiomics without shape features (Reviewer 4, point 7).** Physical voxel geometry cannot be recovered
  (the public PLC-CECT NIfTI files store 1×1×1 mm), so all 56 shape features are removed. The Week-5 selection
  recipe is repeated inside each training fold on the remaining 372 features, then CNN+radiomics and full fusion are
  refitted. **This becomes the main model**; the submitted with-shape model is reported as historical.

**Fixed in advance:** same splits, seed 42, same LightGBM and logistic-regression settings, unknown sex kept as
missing (submitted primary analysis). Primary comparison: full fusion vs CNN+radiomics, paired bootstrap of the
held-out macro-AUC difference (2,000 resamples). Every result is reported whatever its direction.

**Safety:** inputs are only read and are checked against their SHA-256 hashes. All outputs go to a new folder
`liver_project/revision_2026-10/E1_E2_<time>/`; nothing existing is overwritten.

Run the cells in order (Runtime → Run all). A CPU runtime is enough; it takes a few minutes.

In [ ]:
# Cell 1 — Setup: paths, input integrity check, library versions, new output folder
import os, sys, json, hashlib, platform, datetime, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', message='X does not have valid feature names')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount(os.environ['COLAB_DRIVE_MOUNT'])  # see docs/revision_notebooks.md
    ROOT = Path(os.environ['LIVER_PROJECT_ROOT'])
else:
    ROOT = Path(os.environ['E1E2_TEST_ROOT'])  # local test harness only

PROJECT_DIR = ROOT / 'liver_project'
DATA_DIR = ROOT / 'liver_data'
REVISION_DIR = PROJECT_DIR / 'revision_2026-10'
AUDIT_DIR = PROJECT_DIR / 'audit_outputs' / 'sex_correction_primary_20260718T174220Z'
if not REVISION_DIR.is_dir():
    raise FileNotFoundError(f'Revision folder not found: {REVISION_DIR}')

SEED = 42
N_BOOT = 2000
CLASSES = ['HCC', 'ICC', 'CHCC']
LABEL_MAP = {name: index for index, name in enumerate(CLASSES)}

# Every input with the SHA-256 it had when the submitted results were produced (checked 17 Oct 2026).
INPUTS = {
    'patient_data': (DATA_DIR / 'patient_data.csv', '8cd4eb9405a3987c35d7ba4e92893c474f20b6a2791ab23bac2c47ef5532b185'),
    'split': (PROJECT_DIR / 'splits' / 'train_val_test.json', 'cfbe3cb6f9b1901c3d672042e7918a95051d841d328d8353492f91376617513e'),
    'fold_0': (PROJECT_DIR / 'splits' / 'fold_0.json', 'c5f32ebf1c646835055887e258dbc574aedbfea66bb197b4e6999961e255b02b'),
    'fold_1': (PROJECT_DIR / 'splits' / 'fold_1.json', '3726c0f6b915942b11519b4ab1b589fac3590ab4a0a1800b2fa72be8ad3a8391'),
    'fold_2': (PROJECT_DIR / 'splits' / 'fold_2.json', 'f50c29e35f728b7368885f0adf974410007203c0043ea581f2d75f1a2faefe33'),
    'fold_3': (PROJECT_DIR / 'splits' / 'fold_3.json', 'b740859b7bf5e3c3c8ce43707b5805b56ebb35453f732f1cd9a35369f584d37f'),
    'fold_4': (PROJECT_DIR / 'splits' / 'fold_4.json', '32deb2f725b24487ac447a4459d6e072ad90529b7ad7d537b0c73a3f342cd7b8'),
    'oof_meta': (PROJECT_DIR / 'fusion' / 'oof_meta_features.csv', '1587c13b56b921f50a184ebec8951cd4394c4c533f74797caa01bda309f10282'),
    'test_meta': (PROJECT_DIR / 'fusion' / 'test_meta_features.csv', 'cca040d2b212bcde6bffffbe25044ee0f6f119131a7c2c5c27b013a17f1db8c4'),
    'w3_oof': (PROJECT_DIR / 'fusion' / 'week3_cnn_oof.csv', '1c7382e40e3d225005b0d7f5d18780d5a75800639d37feafeecc67a9bca83617'),
    'w3_test': (PROJECT_DIR / 'results' / 'week3_test_predictions.csv', '6341cddfcc060af20c9eea0588cefff045a473291224f333150c6e1119870152'),
    'radiomics_features': (PROJECT_DIR / 'radiomics' / 'radiomics_features.parquet', 'caac3d2cf4f273bd6eebd48154a1dc4bf20adbbb7a896b3c4947a910fd01e156'),
    'w5_cv_predictions': (PROJECT_DIR / 'results' / 'week5_cv_predictions.csv', 'c0a633a089c12f79e87f587b3d773a3f0f30ae57fc7e691104f49cb0ed0c218d'),
    'w5_test_predictions': (PROJECT_DIR / 'results' / 'week5_test_predictions.csv', 'd29fdc9d10d16c4faa6a85f21498ae04682bad02aec371120bc7ac1ee10d3fc4'),
}
for fold in range(5):
    INPUTS[f'w5_features_fold_{fold}'] = (PROJECT_DIR / 'radiomics' / 'models' / f'features_fold_{fold}.json', None)
for name in ['corrected_internal_main_results.csv', 'corrected_internal_bootstrap_ci.csv',
             'corrected_internal_per_class.csv', 'corrected_internal_delong_exact_9_tests.csv']:
    INPUTS['submitted_' + name] = (AUDIT_DIR / name, None)

def sha256_of(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

input_hashes, problems = {}, []
for key, (path, expected) in INPUTS.items():
    if not path.exists():
        problems.append(f'missing: {path}')
        continue
    input_hashes[key] = sha256_of(path)
    if expected is not None and input_hashes[key] != expected:
        problems.append(f'changed since the submitted analysis: {path.name}')
if problems:
    raise RuntimeError('STOP — input check failed:\n' + '\n'.join(problems))

try:
    import lightgbm
except ImportError:   # Colab normally ships LightGBM; install only if it is missing
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm
import sklearn, scipy
VERSIONS = {
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__, 'scipy': scipy.__version__, 'lightgbm': lightgbm.__version__,
    'cpu_count': os.cpu_count(),
}

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = REVISION_DIR / f'E1_E2_{RUN_ID}'
OUT_DIR.mkdir(parents=False, exist_ok=False)   # always a brand-new folder

print('Inputs found and unchanged:', len(input_hashes))
print('Versions:', VERSIONS)
print('Outputs will be written to:', OUT_DIR)

In [ ]:
# Cell 2 — Shared functions, copied from the notebooks that produced the submitted results
#   * metrics, meta-learner, bootstrap, DeLong: final_analysis_SUBMITTED_NUMBERS_sex_correction.ipynb
#   * radiomics feature selection and LightGBM: pipeline-week05-radiomics-model.ipynb
from scipy.stats import norm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import f_classif
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix)
import lightgbm as lgb

def calculate_test_metrics(y_true, probabilities):
    predictions = probabilities.argmax(axis=1)
    return {
        'test_auc': float(roc_auc_score(y_true, probabilities, multi_class='ovr', average='macro')),
        'test_accuracy': float(accuracy_score(y_true, predictions)),
        'test_macro_f1': float(f1_score(y_true, predictions, average='macro', zero_division=0)),
        'test_macro_precision': float(precision_score(y_true, predictions, average='macro', zero_division=0)),
        'test_confusion_matrix': confusion_matrix(y_true, predictions, labels=[0, 1, 2]),
        'test_predictions': predictions,
    }

def evaluate_raw_branch(oof_frame, test_frame, probability_columns):
    fold_rows = []
    for fold_index in range(5):
        fold_frame = oof_frame.loc[oof_frame['fold'].astype(int).eq(fold_index)]
        y_fold = fold_frame['label'].astype(int).to_numpy()
        p_fold = fold_frame[probability_columns].to_numpy(dtype=float)
        pred_fold = p_fold.argmax(axis=1)
        fold_rows.append({
            'fold': fold_index,
            'auc': float(roc_auc_score(y_fold, p_fold, multi_class='ovr', average='macro')),
            'accuracy': float(accuracy_score(y_fold, pred_fold)),
            'macro_f1': float(f1_score(y_fold, pred_fold, average='macro', zero_division=0)),
        })
    y_test = test_frame['label'].astype(int).to_numpy()
    test_probabilities = test_frame[probability_columns].to_numpy(dtype=float)
    return {'cv_metrics': pd.DataFrame(fold_rows), 'test_probabilities': test_probabilities,
            'test_labels': y_test, 'test_patient_ids': test_frame['patient_id'].to_numpy(),
            **calculate_test_metrics(y_test, test_probabilities)}

def fit_meta_learner_cv(feature_columns, oof_frame, test_frame):
    fold_rows, fold_test_probabilities = [], []
    oof_probabilities = np.full((len(oof_frame), 3), np.nan, dtype=float)
    for fold_index in range(5):
        training_mask = oof_frame['fold'].astype(int).ne(fold_index)
        validation_mask = oof_frame['fold'].astype(int).eq(fold_index)
        X_training = oof_frame.loc[training_mask, feature_columns].to_numpy(dtype=np.float32)
        y_training = oof_frame.loc[training_mask, 'label'].astype(int).to_numpy()
        X_validation = oof_frame.loc[validation_mask, feature_columns].to_numpy(dtype=np.float32)
        y_validation = oof_frame.loc[validation_mask, 'label'].astype(int).to_numpy()
        X_test = test_frame[feature_columns].to_numpy(dtype=np.float32)
        X_training[X_training <= -1] = np.nan
        X_validation[X_validation <= -1] = np.nan
        X_test[X_test <= -1] = np.nan
        imputer = SimpleImputer(strategy='median')
        X_training = imputer.fit_transform(X_training)
        X_validation = imputer.transform(X_validation)
        X_test = imputer.transform(X_test)
        if X_training.shape[1] != len(feature_columns):
            raise RuntimeError('An all-missing feature was removed by the imputer.')
        scaler = StandardScaler()
        X_training = scaler.fit_transform(X_training)
        X_validation = scaler.transform(X_validation)
        X_test = scaler.transform(X_test)
        model = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=SEED)
        model.fit(X_training, y_training)
        if not np.array_equal(model.classes_, np.array([0, 1, 2])):
            raise RuntimeError(f'Unexpected model class order: {model.classes_}')
        validation_probabilities = model.predict_proba(X_validation)
        validation_predictions = validation_probabilities.argmax(axis=1)
        oof_probabilities[np.flatnonzero(validation_mask.to_numpy())] = validation_probabilities
        fold_rows.append({
            'fold': fold_index,
            'auc': float(roc_auc_score(y_validation, validation_probabilities, multi_class='ovr', average='macro')),
            'accuracy': float(accuracy_score(y_validation, validation_predictions)),
            'macro_f1': float(f1_score(y_validation, validation_predictions, average='macro', zero_division=0)),
        })
        fold_test_probabilities.append(model.predict_proba(X_test))
    if not np.isfinite(oof_probabilities).all():
        raise RuntimeError('OOF probability matrix is incomplete.')
    test_probabilities = np.mean(fold_test_probabilities, axis=0)
    y_test = test_frame['label'].astype(int).to_numpy()
    return {'cv_metrics': pd.DataFrame(fold_rows), 'oof_probabilities': oof_probabilities,
            'test_probabilities': test_probabilities, 'test_labels': y_test,
            'test_patient_ids': test_frame['patient_id'].to_numpy(),
            **calculate_test_metrics(y_test, test_probabilities)}

def bootstrap_macro_auc(y_true, probabilities, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        indices = rng.integers(0, len(y_true), size=len(y_true))
        y_boot = y_true[indices]
        if len(np.unique(y_boot)) < 3:
            continue
        try:
            values.append(roc_auc_score(y_boot, probabilities[indices], multi_class='ovr', average='macro'))
        except ValueError:
            continue
    values = np.asarray(values, dtype=float)
    return {'bootstrap_mean': float(values.mean()), 'ci_low': float(np.percentile(values, 2.5)),
            'ci_high': float(np.percentile(values, 97.5)), 'successful_resamples': int(len(values))}

def paired_bootstrap_delta(y_true, probabilities_a, probabilities_b, n_boot=N_BOOT, seed=SEED):
    """Held-out macro-AUC(A) - macro-AUC(B), resampling whole patients (same resamples for A and B)."""
    point = (roc_auc_score(y_true, probabilities_a, multi_class='ovr', average='macro')
             - roc_auc_score(y_true, probabilities_b, multi_class='ovr', average='macro'))
    rng = np.random.default_rng(seed)
    deltas = []
    for _ in range(n_boot):
        indices = rng.integers(0, len(y_true), size=len(y_true))
        y_boot = y_true[indices]
        if len(np.unique(y_boot)) < 3:
            continue
        deltas.append(roc_auc_score(y_boot, probabilities_a[indices], multi_class='ovr', average='macro')
                      - roc_auc_score(y_boot, probabilities_b[indices], multi_class='ovr', average='macro'))
    deltas = np.asarray(deltas, dtype=float)
    two_sided = float(min(1.0, 2 * min((deltas <= 0).mean(), (deltas >= 0).mean())))
    return {'delta': float(point), 'ci_low': float(np.percentile(deltas, 2.5)),
            'ci_high': float(np.percentile(deltas, 97.5)), 'bootstrap_two_sided_p': two_sided,
            'successful_resamples': int(len(deltas))}

def compute_midrank(values):
    order = np.argsort(values)
    sorted_values = values[order]
    n_values = len(values)
    midranks = np.zeros(n_values, dtype=np.float64)
    index = 0
    while index < n_values:
        end = index
        while end < n_values and sorted_values[end] == sorted_values[index]:
            end += 1
        midranks[index:end] = 0.5 * (index + end - 1) + 1
        index = end
    output = np.empty(n_values, dtype=np.float64)
    output[order] = midranks
    return output

def delong_test(ground_truth, predictions_one, predictions_two):
    ordering = np.argsort(-ground_truth)
    predictions = np.vstack([predictions_one, predictions_two])[:, ordering]
    positives = int(np.sum(ground_truth == 1))
    negatives = len(ground_truth) - positives
    tx = np.empty((2, positives)); ty = np.empty((2, negatives)); tz = np.empty((2, positives + negatives))
    for model_index in range(2):
        tx[model_index] = compute_midrank(predictions[model_index, :positives])
        ty[model_index] = compute_midrank(predictions[model_index, positives:])
        tz[model_index] = compute_midrank(predictions[model_index])
    aucs = tz[:, :positives].sum(axis=1) / positives / negatives - (positives + 1.0) / 2.0 / negatives
    v01 = (tz[:, :positives] - tx) / negatives
    v10 = 1.0 - (tz[:, positives:] - ty) / positives
    covariance = np.cov(v01) / positives + np.cov(v10) / negatives
    contrast = np.array([1.0, -1.0])
    variance = float(contrast @ covariance @ contrast)
    difference = float(aucs[0] - aucs[1])
    if variance <= 0:
        return {'auc_one': float(aucs[0]), 'auc_two': float(aucs[1]), 'difference': difference, 'p': 1.0}
    z_value = difference / np.sqrt(variance)
    return {'auc_one': float(aucs[0]), 'auc_two': float(aucs[1]), 'difference': difference,
            'p': float(2.0 * norm.sf(abs(z_value)))}

def adjust_pvalues(raw_pvalues):
    raw = np.asarray(raw_pvalues, dtype=float)
    n_tests = len(raw)
    bonferroni = np.minimum(raw * n_tests, 1.0)
    order = np.argsort(raw)
    adjusted = raw[order] * n_tests / np.arange(1, n_tests + 1)
    adjusted = np.minimum(np.minimum.accumulate(adjusted[::-1])[::-1], 1.0)
    benjamini_hochberg = np.empty(n_tests)
    benjamini_hochberg[order] = adjusted
    return bonferroni, benjamini_hochberg

# ---- Week-5 radiomics recipe (unchanged) ----
K_TOP_FEATURES = 30
CORR_THRESHOLD = 0.95

def select_features_train(X_tr, y_tr, feature_names, k=K_TOP_FEATURES, corr_thresh=CORR_THRESHOLD):
    stds = np.nanstd(X_tr, axis=0)
    keep = stds > 1e-10
    X_kept = X_tr[:, keep]
    kept_idx_global = np.where(keep)[0]
    X_nan_to_num = np.nan_to_num(X_kept, nan=np.nanmedian(X_kept))
    corr = np.abs(np.corrcoef(X_nan_to_num.T))
    remove = np.zeros(corr.shape[0], dtype=bool)
    for i in range(corr.shape[0]):
        if remove[i]:
            continue
        for j in range(i + 1, corr.shape[0]):
            if remove[j]:
                continue
            if corr[i, j] > corr_thresh:
                remove[j] = True
    kept_idx_after_corr = kept_idx_global[~remove]
    X_for_f = SimpleImputer(strategy='median').fit_transform(X_tr[:, kept_idx_after_corr])
    F, _ = f_classif(X_for_f, y_tr)
    order = np.argsort(F)[::-1]
    final_idx = kept_idx_after_corr[order[:min(k, len(order))]]
    return final_idx, [feature_names[i] for i in final_idx]

def train_one_fold_radiomics(fold_idx, train_ids, val_ids, df_features, feature_cols):
    df_tr = df_features[df_features['patient_id'].isin(train_ids)].reset_index(drop=True)
    df_va = df_features[df_features['patient_id'].isin(val_ids)].reset_index(drop=True)
    X_tr_all = df_tr[feature_cols].values.astype(np.float32)
    y_tr = df_tr['label'].values
    X_va_all = df_va[feature_cols].values.astype(np.float32)
    y_va = df_va['label'].values
    sel_idx, sel_names = select_features_train(X_tr_all, y_tr, feature_cols)
    X_tr = X_tr_all[:, sel_idx]
    X_va = X_va_all[:, sel_idx]
    imp = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(X_tr)
    X_va = imp.transform(X_va)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, num_leaves=15,
                             min_child_samples=5, reg_alpha=0.1, reg_lambda=0.1,
                             class_weight='balanced', random_state=42, verbose=-1)
    clf.fit(X_tr, y_tr)
    probs = clf.predict_proba(X_va)
    return {'fold': fold_idx, 'auc': float(roc_auc_score(y_va, probs, multi_class='ovr', average='macro')),
            'probs': probs, 'labels': y_va, 'patient_ids': df_va['patient_id'].tolist(),
            'selected_features': sel_names, 'sel_idx': sel_idx, 'imputer': imp, 'scaler': scaler, 'clf': clf}

def run_radiomics(df_features, feature_cols, fold_splits, test_ids):
    """Week-5 recipe: 5 fold models, out-of-fold predictions, 5-model test ensemble."""
    fold_results = [train_one_fold_radiomics(f, s['train_ids'], s['val_ids'], df_features, feature_cols)
                    for f, s in enumerate(fold_splits)]
    oof = pd.concat([pd.DataFrame({'patient_id': r['patient_ids'], 'fold': r['fold'], 'label': r['labels'],
                                   'p_HCC': r['probs'][:, 0], 'p_ICC': r['probs'][:, 1], 'p_CHCC': r['probs'][:, 2]})
                     for r in fold_results], ignore_index=True)
    df_test = df_features[df_features['patient_id'].isin(test_ids)].reset_index(drop=True)
    X_test_all = df_test[feature_cols].values.astype(np.float32)
    probs = np.mean([r['clf'].predict_proba(r['scaler'].transform(r['imputer'].transform(X_test_all[:, r['sel_idx']])))
                     for r in fold_results], axis=0)
    test = pd.DataFrame({'patient_id': df_test['patient_id'].values, 'label': df_test['label'].values,
                         'p_HCC': probs[:, 0], 'p_ICC': probs[:, 1], 'p_CHCC': probs[:, 2]})
    return fold_results, oof, test

MODEL_ORDER = ['clinical_only', 'single_phase_cnn', 'multi_phase_cnn', 'radiomics', 'cnn_plus_radiomics', 'full_fusion']
W4 = ['w4_p_HCC', 'w4_p_ICC', 'w4_p_CHCC']
W5 = ['w5_p_HCC', 'w5_p_ICC', 'w5_p_CHCC']

def six_model_results(oof_frame, test_frame, w3_oof, w3_test):
    return {
        'clinical_only': fit_meta_learner_cv(['age', 'gender'], oof_frame, test_frame),
        'single_phase_cnn': evaluate_raw_branch(w3_oof, w3_test, ['w3_p_HCC', 'w3_p_ICC', 'w3_p_CHCC']),
        'multi_phase_cnn': evaluate_raw_branch(oof_frame, test_frame, W4),
        'radiomics': evaluate_raw_branch(oof_frame, test_frame, W5),
        'cnn_plus_radiomics': fit_meta_learner_cv(W4 + W5, oof_frame, test_frame),
        'full_fusion': fit_meta_learner_cv(W4 + W5 + ['age', 'gender'], oof_frame, test_frame),
    }

def summarize(results, analysis):
    main, folds, per_class, boot = [], [], [], []
    for key in MODEL_ORDER:
        r = results[key]
        cv = r['cv_metrics']['auc']
        main.append({'analysis': analysis, 'model_key': key, 'cv_macro_auc_mean': float(cv.mean()),
                     'cv_macro_auc_sd': float(cv.std(ddof=1)), 'test_macro_auc': r['test_auc'],
                     'test_accuracy': r['test_accuracy'], 'test_macro_f1': r['test_macro_f1'],
                     'test_macro_precision': r['test_macro_precision']})
        for _, row in r['cv_metrics'].iterrows():
            folds.append({'analysis': analysis, 'model_key': key, **row.to_dict()})
        y, p = r['test_labels'], r['test_probabilities']
        recalls = recall_score(y, r['test_predictions'], labels=[0, 1, 2], average=None, zero_division=0)
        precisions = precision_score(y, r['test_predictions'], labels=[0, 1, 2], average=None, zero_division=0)
        for c, name in enumerate(CLASSES):
            per_class.append({'analysis': analysis, 'model_key': key, 'class': name, 'support': int((y == c).sum()),
                              'test_auc': float(roc_auc_score((y == c).astype(int), p[:, c])),
                              'test_recall': float(recalls[c]), 'test_precision': float(precisions[c])})
        b = bootstrap_macro_auc(y, p)
        boot.append({'analysis': analysis, 'model_key': key, 'test_macro_auc_point': r['test_auc'],
                     'ci_95_low': b['ci_low'], 'ci_95_high': b['ci_high'],
                     'successful_resamples': b['successful_resamples']})
    return pd.DataFrame(main), pd.DataFrame(folds), pd.DataFrame(per_class), pd.DataFrame(boot)

def delong_nine(results, analysis):
    y = results['full_fusion']['test_labels']
    fusion = results['full_fusion']['test_probabilities']
    rows = []
    for comparator in ['multi_phase_cnn', 'radiomics', 'cnn_plus_radiomics']:
        for c, name in enumerate(CLASSES):
            d = delong_test((y == c).astype(int), fusion[:, c], results[comparator]['test_probabilities'][:, c])
            rows.append({'analysis': analysis, 'class': name, 'comparator': comparator,
                         'auc_full_fusion': d['auc_one'], 'auc_comparator': d['auc_two'], 'uncorrected_p': d['p']})
    table = pd.DataFrame(rows)
    table['bonferroni_p_9'], table['benjamini_hochberg_p_9'] = adjust_pvalues(table['uncorrected_p'])
    return table

print('Functions ready.')

In [ ]:
# Cell 3 — Load the locked data and build the submitted (corrected-sex) fusion tables
split = json.loads(INPUTS['split'][0].read_text())
dev_ids = [str(x) for x in split['dev_ids']]
test_ids = [str(x) for x in split['test_ids']]
fold_splits = [json.loads(INPUTS[f'fold_{f}'][0].read_text()) for f in range(5)]
assert len(dev_ids) == 222 and len(test_ids) == 56 and not set(dev_ids) & set(test_ids)
for s in fold_splits:
    assert not set(s['train_ids']) & set(s['val_ids'])
    assert set(s['train_ids']) | set(s['val_ids']) == set(dev_ids)

# Sex exactly as in the submitted analysis: male=1, female=0, unresolved=missing (first manifest row per patient)
raw = pd.read_csv(INPUTS['patient_data'][0])
raw_literal = pd.read_csv(INPUTS['patient_data'][0], dtype=str, keep_default_na=False)
raw['patient_id'] = raw['patient_id'].astype(str)
cohort = raw['patient_id'].isin(set(dev_ids + test_ids))
first = raw.loc[cohort].drop_duplicates('patient_id', keep='first').index
token = raw_literal.loc[first, 'gender'].astype(str).str.strip().str.lower()
revised_sex = pd.Series(np.select([token.eq('male'), token.eq('female')], [1.0, 0.0], default=np.nan),
                        index=raw.loc[first, 'patient_id'].values)
assert (revised_sex == 1).sum() == 185 and (revised_sex == 0).sum() == 90 and revised_sex.isna().sum() == 3

oof_meta = pd.read_csv(INPUTS['oof_meta'][0], dtype={'patient_id': str})
test_meta = pd.read_csv(INPUTS['test_meta'][0], dtype={'patient_id': str})
assert set(oof_meta['patient_id']) == set(dev_ids) and set(test_meta['patient_id']) == set(test_ids)
assert np.bincount(test_meta['label'].astype(int), minlength=3).tolist() == [19, 20, 17]
oof_corrected, test_corrected = oof_meta.copy(), test_meta.copy()
oof_corrected['gender'] = oof_corrected['patient_id'].map(revised_sex)
test_corrected['gender'] = test_corrected['patient_id'].map(revised_sex)
assert oof_corrected['gender'].isna().sum() == 1 and test_corrected['gender'].isna().sum() == 2

w3_oof = pd.read_csv(INPUTS['w3_oof'][0], dtype={'patient_id': str})
w3_test = (test_corrected[['patient_id', 'label']]
           .merge(pd.read_csv(INPUTS['w3_test'][0], dtype={'patient_id': str})
                  .rename(columns={'prob_HCC': 'w3_p_HCC', 'prob_ICC': 'w3_p_ICC', 'prob_CHCC': 'w3_p_CHCC'}),
                  on='patient_id', how='inner', validate='one_to_one'))
assert (w3_test['label'].astype(int).values == w3_test['true_label'].astype(int).values).all()
assert (w3_oof.set_index('patient_id').loc[oof_corrected['patient_id'], 'fold'].values
        == oof_corrected['fold'].values).all()
print('Locked data loaded: 222 development / 56 held-out patients, 5 folds, sex 185/90/3 (male/female/unknown).')

In [ ]:
# Cell 4 — E1a: reproduce the submitted internal results (all six models) and compare with the saved tables
results_submitted = six_model_results(oof_corrected, test_corrected, w3_oof, w3_test)
e1_main, e1_folds, e1_per_class, e1_boot = summarize(results_submitted, 'E1_reproduction')
e1_delong = delong_nine(results_submitted, 'E1_reproduction')

saved_main = pd.read_csv(INPUTS['submitted_corrected_internal_main_results.csv'][0])
saved_boot = pd.read_csv(INPUTS['submitted_corrected_internal_bootstrap_ci.csv'][0])
saved_class = pd.read_csv(INPUTS['submitted_corrected_internal_per_class.csv'][0])
saved_delong = pd.read_csv(INPUTS['submitted_corrected_internal_delong_exact_9_tests.csv'][0])

TOLERANCE = 5e-4   # the tolerance the submitted analysis used for its own reproduction checks
checks = []
def check(name, observed, expected):
    checks.append({'check': name, 'observed': float(observed), 'submitted': float(expected),
                   'abs_diff': abs(float(observed) - float(expected))})

for _, row in saved_main.iterrows():
    ours = e1_main.set_index('model_key').loc[row['model_key']]
    for column in ['cv_macro_auc_mean', 'cv_macro_auc_sd', 'test_macro_auc', 'test_accuracy',
                   'test_macro_f1', 'test_macro_precision']:
        check(f"{row['model_key']} {column}", ours[column], row[column])
for _, row in saved_boot.iterrows():
    ours = e1_boot.set_index('model_key').loc[row['model_key']]
    check(f"{row['model_key']} CI low", ours['ci_95_low'], row['ci_95_low'])
    check(f"{row['model_key']} CI high", ours['ci_95_high'], row['ci_95_high'])
for _, row in saved_class.iterrows():
    ours = e1_per_class.set_index(['model_key', 'class']).loc[(row['model_key'], row['class'])]
    check(f"{row['model_key']} {row['class']} AUC", ours['test_auc'], row['test_auc'])
for _, row in saved_delong.iterrows():
    comparator_key = row['comparison'].split(' vs ')[1]   # e.g. 'full_fusion vs multi_phase_cnn'
    ours = e1_delong.set_index(['comparator', 'class']).loc[(comparator_key, row['class'])]
    check(f"DeLong {comparator_key} {row['class']} p", ours['uncorrected_p'], row['uncorrected_p'])

e1_checks = pd.DataFrame(checks)
E1A_PASS = bool((e1_checks['abs_diff'] <= TOLERANCE).all())
print(f"E1a: {len(e1_checks)} submitted values compared; largest difference = {e1_checks['abs_diff'].max():.2e}")
print(e1_main[['model_key', 'cv_macro_auc_mean', 'test_macro_auc']].round(4).to_string(index=False))
if not E1A_PASS:
    print(e1_checks.sort_values('abs_diff', ascending=False).head(10).to_string(index=False))
    raise RuntimeError('STOP — E1a did not reproduce the submitted results. Send this output for review.')
print('E1a PASS: the submitted internal results are reproduced.')

In [ ]:
# Cell 5 — E1b: re-run the Week-5 radiomics recipe (with shape features) and compare with the saved Week-5 outputs
df_features = pd.read_parquet(INPUTS['radiomics_features'][0])
df_features['patient_id'] = df_features['patient_id'].astype(str)
feature_cols = [c for c in df_features.columns if c not in ('patient_id', 'label')]
assert len(feature_cols) == 428 and len(df_features) == 278
values = df_features[feature_cols].values.astype(np.float32)
if np.isinf(values).any():
    values[np.isinf(values)] = np.nan
    df_features.loc[:, feature_cols] = values

folds_with, oof_with, test_with = run_radiomics(df_features, feature_cols, fold_splits, test_ids)

# (1) the selected features must be identical to the saved fold models
feature_match = []
for f, r in enumerate(folds_with):
    saved = json.loads(INPUTS[f'w5_features_fold_{f}'][0].read_text())['sel_names']
    feature_match.append(list(r['selected_features']) == list(saved))
# (2) probabilities and fold AUCs compared with the saved Week-5 predictions
saved_oof = oof_meta.set_index('patient_id')[W5]
ours_oof = oof_with.set_index('patient_id').loc[saved_oof.index, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
oof_diff = float(np.abs(ours_oof - saved_oof.to_numpy()).max())
saved_test = (pd.read_csv(INPUTS['w5_test_predictions'][0], dtype={'patient_id': str})
              .set_index('patient_id')[['prob_HCC', 'prob_ICC', 'prob_CHCC']])
ours_test = test_with.set_index('patient_id').loc[saved_test.index, ['p_HCC', 'p_ICC', 'p_CHCC']].to_numpy()
test_diff = float(np.abs(ours_test - saved_test.to_numpy()).max())
fold_auc_ours = [r['auc'] for r in folds_with]

E1B_FEATURES_IDENTICAL = all(feature_match)
E1B_EXACT = E1B_FEATURES_IDENTICAL and oof_diff < 1e-6 and test_diff < 1e-6
E1B_CLOSE = E1B_FEATURES_IDENTICAL and max(oof_diff, test_diff) < 1e-3
print('Selected features identical to saved models, per fold:', feature_match)
print(f'Largest probability difference vs saved Week 5: OOF {oof_diff:.2e}, held-out {test_diff:.2e}')
print('Fold AUCs (re-run):', [round(a, 4) for a in fold_auc_ours], ' saved: [0.8891, 0.8876, 0.9888, 0.9429, 0.8995]')
if not E1B_FEATURES_IDENTICAL:
    raise RuntimeError('STOP — feature selection did not reproduce the saved models. Send this output for review.')
if E1B_EXACT:
    print('E1b PASS: Week-5 radiomics reproduced exactly.')
elif E1B_CLOSE:
    print('E1b PASS (close): same features; LightGBM probabilities differ by less than 0.001 from May 2026.')
else:
    print('E1b WARNING: same features, but LightGBM probabilities differ noticeably from May 2026')
    print('             (probably a different LightGBM version). The notebook continues, and E2 is compared with')
    print('             this same-session with-shape re-run, so only the shape removal differs.')
    print('             Send this output for review before using E2 numbers in the paper.')

In [ ]:
# Cell 6 — E2: radiomics WITHOUT shape features (same recipe, inside each training fold)
shape_cols = [c for c in feature_cols if '_original_shape_' in c]
noshape_cols = [c for c in feature_cols if '_original_shape_' not in c]
assert len(shape_cols) == 56 and len(noshape_cols) == 372

folds_noshape, oof_noshape, test_noshape = run_radiomics(df_features, noshape_cols, fold_splits, test_ids)
assert not any('_shape_' in name for r in folds_noshape for name in r['selected_features'])

def radiomics_fold_table(fold_results, label):
    return pd.DataFrame([{'model': label, 'fold': r['fold'], 'auc': r['auc'],
                          'n_shape_selected': sum('_shape_' in n for n in r['selected_features'])}
                         for r in fold_results])

e2_radiomics_folds = pd.concat([radiomics_fold_table(folds_with, 'with_shape_rerun'),
                                radiomics_fold_table(folds_noshape, 'no_shape')], ignore_index=True)
print(e2_radiomics_folds.pivot(index='fold', columns='model', values='auc').round(4).to_string())
print(f"Radiomics CV macro-AUC: with shape {np.mean([r['auc'] for r in folds_with]):.4f}"
      f"  ->  without shape {np.mean([r['auc'] for r in folds_noshape]):.4f}")

In [ ]:
# Cell 7 — E2: refit CNN+radiomics and full fusion with the new radiomics probabilities
def with_radiomics(frame, radiomics_predictions):
    mapped = radiomics_predictions.set_index('patient_id').loc[frame['patient_id'], ['p_HCC', 'p_ICC', 'p_CHCC']]
    out = frame.copy()
    out[W5] = mapped.to_numpy()
    return out

oof_e2, test_e2 = with_radiomics(oof_corrected, oof_noshape), with_radiomics(test_corrected, test_noshape)
oof_ref, test_ref = with_radiomics(oof_corrected, oof_with), with_radiomics(test_corrected, test_with)

results_e2 = six_model_results(oof_e2, test_e2, w3_oof, w3_test)            # NEW MAIN MODEL
results_ref = six_model_results(oof_ref, test_ref, w3_oof, w3_test)         # same-session with-shape reference
e2_main, e2_folds, e2_per_class, e2_boot = summarize(results_e2, 'E2_no_shape_main')
ref_main, ref_folds, ref_per_class, ref_boot = summarize(results_ref, 'with_shape_same_session')
e2_delong = delong_nine(results_e2, 'E2_no_shape_main')

comparison = (e1_main.set_index('model_key')[['cv_macro_auc_mean', 'test_macro_auc']]
              .rename(columns=lambda c: 'submitted_' + c)
              .join(e2_main.set_index('model_key')[['cv_macro_auc_mean', 'cv_macro_auc_sd', 'test_macro_auc',
                                                     'test_accuracy', 'test_macro_f1']].rename(columns=lambda c: 'E2_' + c))
              .join(e2_boot.set_index('model_key')[['ci_95_low', 'ci_95_high']].rename(columns=lambda c: 'E2_' + c)))
print(comparison.round(4).to_string())

In [ ]:
# Cell 8 — Primary prespecified comparison and secondary (descriptive) comparisons
y_test = results_e2['full_fusion']['test_labels']
rows = []
def add_delta(name, role, a, b):
    d = paired_bootstrap_delta(y_test, a, b)
    rows.append({'comparison': name, 'role': role, **d})

add_delta('E2 full fusion - E2 CNN+radiomics', 'PRIMARY (prespecified)',
          results_e2['full_fusion']['test_probabilities'], results_e2['cnn_plus_radiomics']['test_probabilities'])
add_delta('E2 radiomics (no shape) - radiomics with shape (same session)', 'secondary',
          results_e2['radiomics']['test_probabilities'], results_ref['radiomics']['test_probabilities'])
add_delta('E2 full fusion - full fusion with shape (same session)', 'secondary',
          results_e2['full_fusion']['test_probabilities'], results_ref['full_fusion']['test_probabilities'])
add_delta('E2 full fusion - multiphase CNN', 'secondary',
          results_e2['full_fusion']['test_probabilities'], results_e2['multi_phase_cnn']['test_probabilities'])
deltas = pd.DataFrame(rows)

cv_pairs = []
for name, a, b in [('full fusion - CNN+radiomics (E2)', results_e2['full_fusion'], results_e2['cnn_plus_radiomics']),
                   ('radiomics no shape - with shape', results_e2['radiomics'], results_ref['radiomics'])]:
    diff = a['cv_metrics']['auc'].to_numpy() - b['cv_metrics']['auc'].to_numpy()
    cv_pairs.append({'comparison': name, 'mean_fold_difference': float(diff.mean()),
                     'folds_positive': int((diff > 0).sum()), 'fold_differences': np.round(diff, 4).tolist()})
cv_pairs = pd.DataFrame(cv_pairs)

print(deltas.round(4).to_string(index=False))
print()
print(cv_pairs.to_string(index=False))
primary = deltas.iloc[0]
print(f"\nPRIMARY: full fusion minus CNN+radiomics = {primary['delta']:+.4f} "
      f"(95% CI {primary['ci_low']:+.4f} to {primary['ci_high']:+.4f}).")
print('Interpretation rule (fixed in advance): a CI that includes 0 means no evidence that fusion differs.')

In [ ]:
# Cell 9 — Save everything to the new output folder (nothing else is written)
import joblib
model_dir = OUT_DIR / 'e2_radiomics_noshape_fold_models'
model_dir.mkdir()
for r in folds_noshape:
    joblib.dump({'clf': r['clf'], 'imputer': r['imputer'], 'scaler': r['scaler'], 'sel_idx': r['sel_idx'],
                 'sel_names': r['selected_features'], 'feature_pool': noshape_cols},
                model_dir / f"fold_{r['fold']}.joblib")

tables = {
    'E1a_reproduction_checks.csv': e1_checks,
    'E1a_main_results.csv': e1_main,
    'E2_radiomics_fold_auc_with_vs_without_shape.csv': e2_radiomics_folds,
    'E2_radiomics_noshape_oof_predictions.csv': oof_noshape,
    'E2_radiomics_noshape_test_predictions.csv': test_noshape,
    'E2_radiomics_withshape_rerun_oof_predictions.csv': oof_with,
    'E2_radiomics_withshape_rerun_test_predictions.csv': test_with,
    'E2_fusion_oof_meta_features_noshape.csv': oof_e2,
    'E2_fusion_test_meta_features_noshape.csv': test_e2,
    'E2_main_results.csv': pd.concat([e2_main, ref_main], ignore_index=True),
    'E2_fold_metrics.csv': pd.concat([e2_folds, ref_folds], ignore_index=True),
    'E2_per_class.csv': pd.concat([e2_per_class, ref_per_class], ignore_index=True),
    'E2_bootstrap_ci.csv': pd.concat([e2_boot, ref_boot], ignore_index=True),
    'E2_delong_9_tests_exploratory.csv': e2_delong,
    'E2_paired_bootstrap_deltas.csv': deltas,
    'E2_cv_paired_fold_differences.csv': cv_pairs,
    'E2_vs_submitted.csv': comparison.reset_index(),
}
for name, table in tables.items():
    table.to_csv(OUT_DIR / name, index=False)
(OUT_DIR / 'E2_selected_features_noshape.json').write_text(json.dumps(
    {f"fold_{r['fold']}": r['selected_features'] for r in folds_noshape}, indent=2))

summary = {
    'run_id': RUN_ID, 'notebook': 'E1_E2_reproduce_and_noshape_radiomics.ipynb', 'versions': VERSIONS,
    'input_sha256': input_hashes,
    'E1a_pass': E1A_PASS, 'E1a_max_abs_diff': float(e1_checks['abs_diff'].max()),
    'E1b_features_identical': E1B_FEATURES_IDENTICAL, 'E1b_exact': E1B_EXACT, 'E1b_close': E1B_CLOSE,
    'E1b_max_prob_diff_oof': oof_diff, 'E1b_max_prob_diff_test': test_diff,
    'E2_feature_pool': {'all': 428, 'shape_removed': len(shape_cols), 'remaining': len(noshape_cols)},
    'E2_main_results': e2_main.drop(columns='analysis').to_dict(orient='records'),
    'E2_primary_comparison': deltas.iloc[0].to_dict(),
    'protocol': {'splits': 'unchanged', 'seed': SEED, 'bootstrap_resamples': N_BOOT,
                 'sex_handling': 'unknown kept missing, fold-median imputation (submitted primary)',
                 'cnn_models': 'unchanged (saved W3/W4 predictions)', 'test_set_used_for_tuning': False},
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, default=str))

manifest = [{'file': p.relative_to(OUT_DIR).as_posix(), 'bytes': p.stat().st_size, 'sha256': sha256_of(p)}
            for p in sorted(OUT_DIR.rglob('*')) if p.is_file()]
pd.DataFrame(manifest).to_csv(OUT_DIR / 'output_manifest.csv', index=False)

print(f'Saved {len(manifest)} files to {OUT_DIR}')
print('\n================ COPY FROM HERE AND SEND BACK ================')
print(f'Run {RUN_ID} | E1a pass: {E1A_PASS} | E1b features identical: {E1B_FEATURES_IDENTICAL} '
      f'| E1b exact: {E1B_EXACT} | E1b close: {E1B_CLOSE}')
print(f'E1b max prob diff: OOF {oof_diff:.2e}, test {test_diff:.2e}')
print(comparison.round(4).to_string())
print(deltas[['comparison', 'delta', 'ci_low', 'ci_high']].round(4).to_string(index=False))
print('Versions:', VERSIONS)
print('==============================================================')